# Filtering Groups with `.filter()`

Normally, when we filter a DataFrame, we filter individual rows (e.g., `df[df['Sales'] > 100]`). But sometimes, we want to filter out **entire groups** based on a property of the group as a whole.

The **`.filter()`** method applied to a GroupBy object allows you to do exactly this. You pass a function that returns a boolean value (`True` or `False`) based on a group-level aggregation (like group sum or group count). If a group meets the condition, all of its original rows are kept; if not, the entire group is discarded.


### 2. Real-World Analogy
Imagine a soccer tournament.
*   **Row-level filtering**: Discarding individual players who scored 0 goals.
*   **Group-level filtering (`.filter()`)**: Eliminating *entire teams* from the tournament if their total team goals are less than 5. Even if a team has an individual superstar player, the entire team is dropped because the collective team score did not meet the requirement.

### Code Examples

Let's use a dataset tracking vehicle sales from different manufacturers:




In [1]:
import pandas as pd

cars = pd.DataFrame({
    'Manufacturer': ['Audi', 'Audi', 'Volvo', 'Volvo', 'Ford', 'Ford', 'Ford'],
    'Model': ['A4', 'A6', 'S60', 'XC90', 'Focus', 'Fiesta', 'Mustang'],
    'Sales_Thousands': [40, 15, 25, 20, 80, 55, 30]
})

print("--- Original Sales Data ---")
print(cars)

--- Original Sales Data ---
  Manufacturer    Model  Sales_Thousands
0         Audi       A4               40
1         Audi       A6               15
2        Volvo      S60               25
3        Volvo     XC90               20
4         Ford    Focus               80
5         Ford   Fiesta               55
6         Ford  Mustang               30


#### Filtering Out Low-Volume Manufacturers
We want to keep only the manufacturers whose **total sales** across all models are greater than 50 thousand.

Let's write a custom filter function and apply it:

In [2]:
# Custom filter function
# The parameter 'x' represents the sub-dataframe of each group (each manufacturer) [691]
def filter_by_sales(x):
    return x['Sales_Thousands'].sum() > 50

# Apply the filter [690, 692]
filtered_cars = cars.groupby('Manufacturer').filter(filter_by_sales)
print("--- Filtered DataFrame ---")
print(filtered_cars)

--- Filtered DataFrame ---
  Manufacturer    Model  Sales_Thousands
0         Audi       A4               40
1         Audi       A6               15
4         Ford    Focus               80
5         Ford   Fiesta               55
6         Ford  Mustang               30



*Why was Volvo removed?*
*   **Ford's** total sales = 80 + 55 + 30 = 165 (> 50) ➡️ **KEPT**
*   **Audi's** total sales = 40 + 15 = 55 (> 50) ➡️ **KEPT**
*   **Volvo's** total sales = 25 + 20 = 45 (<= 50) ➡️ **DISCARDED**

Every single row associated with Volvo was dropped from the final DataFrame!


### Common Pitfalls to Avoid
1.  **Returning Non-Boolean Values**: The function passed to `.filter()` must return a single `True` or `False` for the entire group. Returning a number or a Series of booleans will result in a `TypeError`.
2.  **Confusing `.filter()` on DataFrames vs GroupBy**: `df.filter()` selects columns or index labels by name. `df.groupby().filter()` filters groups based on collective conditions. They are completely different operations.


#### Exercise 1 (Medium)
You have a dataset of departments and employee projects:
```python
import pandas as pd
company_projects = pd.DataFrame({
    'Department': ['HR', 'HR', 'IT', 'IT', 'Marketing', 'Marketing'],
    'Project': ['Recruiting', 'Onboarding', 'Security', 'Database', 'Ad_Campaign', 'SEO'],
    'Budget_USD': [10000, 15000, 80000, 50000, 12000, 8000]
})
```
Filter the DataFrame to keep only departments whose **average budget** across all projects is greater than $20,000.


In [3]:
import pandas as pd

company_projects = pd.DataFrame({
    'Department': ['HR', 'HR', 'IT', 'IT', 'Marketing', 'Marketing'],
    'Project': ['Recruiting', 'Onboarding', 'Security', 'Database', 'Ad_Campaign', 'SEO'],
    'Budget_USD': [10000, 15000, 80000, 50000, 12000, 8000]
})

# Filter departments with average budget > 20,000
high_budget_depts = company_projects.groupby('Department').filter(lambda x: x['Budget_USD'].mean() > 20000)
print(high_budget_depts)

  Department   Project  Budget_USD
2         IT  Security       80000
3         IT  Database       50000


*Explanation:*
*   **HR average**: (10,000 + 15,000) / 2 = 12,500 (Dropped)
*   **IT average**: (80,000 + 50,000) / 2 = 65,000 (Kept)
*   **Marketing average**: (12,000 + 8,000) / 2 = 10,000 (Dropped)
